# 运行时


​
## 概述
LangChaincreate_agent底层运行在LangGraph的运行时环境中。
LangGraph 公开了一个Runtime包含以下信息的对象：
- 上下文：静态信息，例如用户 ID、数据库连接或其他代理调用依赖项。
- 存储：用于长期记忆的BaseStore实例
- 流写入器"custom"：用于通过流模式传输信息的对象
- 您可以在工具和中间件中访问运行时信息。
​


## 使用权
使用 create_agent 创建代理时，可以指定 context_schema，以定义存储在代理 Runtime中的上下文结构。

调用代理时，将包含运行相关配置的context参数传递给代理：

In [ ]:
from dataclasses import dataclass

from langchain.agents import create_agent


@dataclass
class Context:
    user_name: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    context_schema=Context  
)

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")  
)

## 中间件内部
您可以访问中间件中的运行时信息，以根据用户上下文创建动态提示、修改消息或控制代理行为。
用于在中间件装饰器中request.runtime访问Runtime对象。运行时对象可通过ModelRequest传递给中间件函数的参数访问。

In [ ]:
from dataclasses import dataclass

from langchain.messages import AnyMessage
from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import dynamic_prompt, ModelRequest, before_model, after_model
from langgraph.runtime import Runtime


@dataclass
class Context:
    user_name: str

# Dynamic prompts
@dynamic_prompt
def dynamic_system_prompt(request: ModelRequest) -> str:
    user_name = request.runtime.context.user_name  
    system_prompt = f"You are a helpful assistant. Address the user as {user_name}."
    return system_prompt

# Before model hook
@before_model
def log_before_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:  
    print(f"Processing request for user: {runtime.context.user_name}")  
    return None

# After model hook
@after_model
def log_after_model(state: AgentState, runtime: Runtime[Context]) -> dict | None:  
    print(f"Completed request for user: {runtime.context.user_name}")  
    return None

agent = create_agent(
    model="gpt-5-nano",
    tools=[...],
    middleware=[dynamic_system_prompt, log_before_model, log_after_model],  
    context_schema=Context
)

agent.invoke(
    {"messages": [{"role": "user", "content": "What's my name?"}]},
    context=Context(user_name="John Smith")
)